# Pipeline de prétraitement

In [7]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer,
    OrdinalEncoder,
)
import joblib


PROJECT_DIR = Path().cwd().parent.resolve()
DATA_DIR = PROJECT_DIR / "data"
DATA_PATH = DATA_DIR / "03_DONNEES.csv"
OUTPUT_DIR = PROJECT_DIR / "output"
MODELS_DIR = OUTPUT_DIR / "models"
NAIVE_PREPROCESSOR_PATH = MODELS_DIR / "naive_preprocessor.pkl"
FEATURE_ENGINEERING_PREPROCESSOR_PATH = (
    MODELS_DIR / "feature_engineering_preprocessor.pkl"
)
ENRICHED_DATAFRAME_PATH = DATA_DIR / "enriched_dataset.csv"

## Chargement des données

In [8]:
df = pd.read_csv(DATA_PATH.as_posix())

X = df.drop(["customerID", "Churn"], axis=1)
y = df["Churn"].copy()

## Création des pipelines de prétraitement

In [9]:
cat_features = X.select_dtypes(include=["object", "str"]).columns.to_list()
num_features = X.select_dtypes(include=np.number).columns.to_list()

print("Features catégorielles : ", cat_features)
print("Features numériques : ", num_features)

Features catégorielles :  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract']
Features numériques :  ['SeniorCitizen', 'tenure', 'InternetCharges', 'MonthlyCharges', 'TotalCharges']


In [10]:
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

naive_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_features),
        ("num", num_pipeline, num_features),
    ]
)

In [11]:
enriched_df = df.copy()

service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

enriched_df["NumberServices"] = enriched_df[service_cols].eq("Yes").sum(axis=1)

enriched_df["AverageMonthly"] = enriched_df["TotalCharges"] / enriched_df["tenure"]
enriched_df["AverageMonthly"] = enriched_df["AverageMonthly"].fillna(
    enriched_df["MonthlyCharges"]
)


def tenure_bucket(t):
    if t <= 6:
        return "client_novice"
    elif t <= 24:
        return "client_adepte"
    else:
        return "client_confirme"


enriched_df["TenureSegment"] = enriched_df["tenure"].apply(tenure_bucket)

enriched_df["ChargePerService"] = enriched_df["MonthlyCharges"] / (
    enriched_df["NumberServices"] + 1
)

In [12]:
cat_features_selected = [
    "gender",
    "Partner",
    "Dependents",  # Distribution du Churn par modalité très proche de celle de dataset (0.4 points)
    "PhoneService",
    "MultipleLines",
    # "InternetService",  # Redondant avec InternetCharges et les services internets
    "OnlineSecurity",  # Distribution du Churn par modalité très proche de celle de dataset (0.2 points)
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",  # Distribution du Churn par modalité très proche de celle de dataset (0.4 points)
    "StreamingMovies",
    # "Contract",  # feature ordinale
]
ordinal_feature = [
    "Contract",
    "TenureSegment",
]
num_features_selected = [
    "SeniorCitizen",  # Distribution du Churn par modalité très proche de celle de dataset (0.1 points)
    "tenure",
    # "MonthlyCharges",  # Redondant avec InternetCharges
    "InternetCharges",
    # "TotalCharges",  # Transformation x -> (1+x)^{0.33}
    "NumberServices",
    "AverageMonthly",
    "ChargePerService",
]
power_feature = [
    "TotalCharges",
]

print("Features catégorielles sélectionnées : ", cat_features_selected)
print("Features ordinale sélectionnées : ", ordinal_feature)
print("Features numériques transformées : ", power_feature)
print("Features numériques sélectionnées : ", num_features_selected)


contract_order = ["Month-to-month", "One year", "Two year"]
tenure_segment_order = ["client_novice", "client_adepte", "client_confirme"]

ordinal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=[contract_order, tenure_segment_order],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

alpha = 0.33


def power_transform(x):
    return np.power(1 + x, alpha)


def inverse_power_transform(y):
    return np.power(y, 1 / alpha) - 1


power_pipeline = Pipeline(
    [
        (
            "power",
            FunctionTransformer(
                func=power_transform,
                inverse_func=inverse_power_transform,
                feature_names_out="one-to-one",
                validate=True,
            ),
        ),
        ("scaler", StandardScaler()),
    ]
)

feature_engineering_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_features),
        ("ord", ordinal_pipeline, ordinal_feature),
        ("power", power_pipeline, power_feature),
        ("num", num_pipeline, num_features),
    ]
)

Features catégorielles sélectionnées :  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
Features ordinale sélectionnées :  ['Contract', 'TenureSegment']
Features numériques transformées :  ['TotalCharges']
Features numériques sélectionnées :  ['SeniorCitizen', 'tenure', 'InternetCharges', 'NumberServices', 'AverageMonthly', 'ChargePerService']


## Sauvegarde des pipelines de prétraitement et du dataset enrichie

In [13]:
enriched_df.to_csv(ENRICHED_DATAFRAME_PATH.as_posix(), index=False)
print("Enriched DataFrame path: ", ENRICHED_DATAFRAME_PATH.as_posix())

joblib.dump(naive_preprocessor, NAIVE_PREPROCESSOR_PATH.as_posix())
print("Naive preprocessor pipeline path: ", NAIVE_PREPROCESSOR_PATH.as_posix())

joblib.dump(
    feature_engineering_preprocessor, FEATURE_ENGINEERING_PREPROCESSOR_PATH.as_posix()
)
print(
    "Feature engineering preprocessor pipeline path: ",
    FEATURE_ENGINEERING_PREPROCESSOR_PATH.as_posix(),
)

Enriched DataFrame path:  C:/Users/Administrateur/Documents/DESSAUX_Damien_ECF3/data/enriched_dataset.csv
Naive preprocessor pipeline path:  C:/Users/Administrateur/Documents/DESSAUX_Damien_ECF3/output/models/naive_preprocessor.pkl
Feature engineering preprocessor pipeline path:  C:/Users/Administrateur/Documents/DESSAUX_Damien_ECF3/output/models/feature_engineering_preprocessor.pkl
